# extras

> Account credit balance, browser attach links, shell commands, directory sync, and one-shot runners for Colab

- skip_exec: true

These helpers extend `fastcolab.core`. Two call Colab's older `/tun/m/` endpoints for account balances and browser links absent from `v1beta`. The rest handle shell commands, hardware checks, directory syncing, remote notebooks, and ephemeral scripts.


In [ ]:
#| default_exp extras

In [ ]:
#| hide
from nbdev.showdoc import *
from fastcore.test import *
import os, tempfile

In [ ]:
#| export
import asyncio, base64, fnmatch, io, json, tarfile, uuid, webbrowser
from pathlib import Path
from urllib.parse import quote, urlparse

from fastcore.nbio import Notebook, read_nb, write_nb
from fastcore.utils import *
from fastgws.auth import refresh_creds
from jupyasyncclient.http import http_client
from fastcolab.core import Colab, ColabRuntime, outs_text

In [ ]:
#| hide
from fastcore.nbio import mk_cell, new_nb

## Live compute unit balance and burn rate

Colab's `v1beta` REST API returns only your subscription tier (`SUBSCRIPTION_TIER_PRO_PLUS`). Live credit balance and hourly burn rate live on Colab's older tunnel backend at `https://colab.research.google.com/tun/m/ccu-info`. Every response starts with `)]}'\n` to prevent cross-site script inclusion. `_tunnel_get` refreshes stale credentials, sends `authuser=0` and `X-Colab-Client-Agent`, and strips that prefix.


In [ ]:
#| export
TUNNEL_BASE = 'https://colab.research.google.com'
_XSSI_PREFIX = ")]}'\n"
_CLIENT_AGENT = 'colab-cli'

async def _tunnel_get(
    creds, # `google-auth` credentials with `colaboratory` and `userinfo.email` scopes
    path:str, # Path beneath `/tun/m/`, such as `'ccu-info'`
    params:dict=None, # Extra query parameters
    timeout:float=30, # Request timeout in seconds
)->dict:
    "Send an authenticated GET request to Colab's `/tun/m/` backend and return parsed JSON."
    if creds is None: raise ValueError('Colab client has no credentials attached')
    if not creds.valid: await refresh_creds(creds)
    hdrs = {'Authorization': f'Bearer {creds.token}', 'Accept': 'application/json', 'X-Colab-Client-Agent': _CLIENT_AGENT}
    q = {'authuser': '0', **(params or {})}
    async with http_client(TUNNEL_BASE, headers=hdrs, timeout=timeout) as hc: r = await hc.get(f'/tun/m/{path.lstrip("/")}', params=q)
    if r.status_code in (401, 403):
        raise PermissionError(f'/tun/m/{path} rejected credentials ({r.status_code}); re-authorize with `gclientid-auth <email> --preset colab` to include `userinfo.email`')
    r.raise_for_status()
    txt = r.text[len(_XSSI_PREFIX):] if r.text.startswith(_XSSI_PREFIX) else r.text
    return json.loads(txt) if txt.strip() else {}

`ColabUsage` wraps the raw `ccu-info` fields in short snake_case properties and renders a compact summary in notebooks. `Colab.usage` calls `/tun/m/ccu-info` using the credentials already attached to `Colab`.

In [ ]:
#| export
class ColabUsage(AttrDict):
    "Account-level Colab compute unit balance, hourly burn rate, and eligible hardware."
    @property
    def balance(self)->float: return float(self.get('currentBalance', 0.0))
    @property
    def rate_hourly(self)->float: return float(self.get('consumptionRateHourly', 0.0))
    @property
    def assignments_count(self)->int: return int(self.get('assignmentsCount', 0))
    @property
    def eligible_gpus(self)->list: return list(self.get('eligibleGpus', []))
    @property
    def eligible_tpus(self)->list: return list(self.get('eligibleTpus', []))

    def __repr__(self): return f'ColabUsage(balance={self.balance:.2f}, rate={self.rate_hourly:.2f}/hr, active={self.assignments_count})'
    def _repr_markdown_(self):
        gpus, tpus = ', '.join(self.eligible_gpus) or 'none', ', '.join(self.eligible_tpus) or 'none'
        return f'**{self.balance:.2f} compute units** (`{self.rate_hourly:.2f}/hr`, `{self.assignments_count}` active); GPUs: {gpus}; TPUs: {tpus}'

@patch
async def usage(self:Colab)->ColabUsage:
    "Fetch live compute unit balance, hourly burn rate, and eligible accelerators from `/tun/m/ccu-info`."
    return ColabUsage(await _tunnel_get(self.creds, 'ccu-info'))

Connect to Colab and check the live balance before starting a server. `gclientid`'s `colab` preset already grants `userinfo.email` alongside `colaboratory`. Your existing token works with no extra sign-in.


In [ ]:
acct = os.environ['FASTCOLAB_ACCOUNT']
c = await Colab.connect(account=acct, drive=False)
u0 = await c.usage()
assert u0.balance >= 0
u0

## Opening a live runtime in the browser

Runtimes from `v1beta.runtimes` have `r-...` IDs. Colab's web UI expects the shorter `/tun/m/` backend ID (`m-...`). `endpoint` decodes that ID from the `aud` claim inside `rt.token`'s JWT payload (falling back to the proxy hostname) with no extra request. `browser_url` builds a link with both `?dbu=` and the unencoded `#datalabBackendUrl=` fragment. Opening it attaches your browser directly to the running virtual machine.


In [ ]:
#| export
def _jwt_payload(token:str)->dict:
    "Decode the payload segment of a JWT without signature verification."
    parts = token.split('.')
    if len(parts) < 2: return {}
    pad = '=' * (-len(parts[1]) % 4)
    return json.loads(base64.urlsafe_b64decode(parts[1] + pad))

@patch(as_prop=True)
def endpoint(self:ColabRuntime)->str:
    "Return the `/tun/m/` backend endpoint ID embedded in this runtime's proxy token or URL."
    aud = _jwt_payload(self.token).get('aud')
    if aud: return aud
    host = urlparse(self.url).hostname or ''
    return host.split('-', 1)[1].rsplit('-c.', 1)[0] if '-' in host else self.id

@patch
def browser_url(
    self:ColabRuntime,
    host:str='https://colab.research.google.com', # Colab frontend origin
    open_browser:bool=False, # Also open the URL in the default browser?
)->str:
    "Return a Colab browser URL that attaches directly to this running virtual machine."
    base = host.rstrip('/')
    path = f'/tun/m/{self.endpoint}'
    link = f'{base}/notebooks/empty.ipynb?dbu={quote(path, safe="")}#datalabBackendUrl={base}{path}'
    if open_browser: webbrowser.open(link)
    return link

Start a High-RAM CPU runtime to exercise the rest of this notebook. Calling `usage()` again shows the active assignment count and hourly burn rate update right away.


In [ ]:
rt = await c.start('cpu')
u1 = await c.usage()
assert u1.assignments_count >= u0.assignments_count
assert rt.endpoint.startswith('m-')
link = rt.browser_url()
assert f'/tun/m/{rt.endpoint}' in link and '#datalabBackendUrl=' in link
dict(endpoint=rt.endpoint, rate_hourly=u1.rate_hourly, active=u1.assignments_count, url=link)

## Shell commands and hardware inspection

Running shell commands directly through `rt.exec` means repeating `subprocess.run` boilerplate every time. `sh` wraps that call behind a unique JSON marker so shell output stays separate from kernel noise. It returns `AttrDict(returncode, stdout, stderr)` and raises `RuntimeError` on non-zero exits when `check=True`.


In [ ]:
#| export
async def _exec_json(rt:ColabRuntime, body:str, timeout:float=None)->AttrDict:
    "Run remote Python `body` that assigns `_res` (JSON-serializable) and parse it back."
    tag = f'__FC_{uuid.uuid4().hex}__'
    code = f'{body}\nimport json as _fc_j\nprint({tag!r} + _fc_j.dumps(_res) + {tag!r})\n'
    outs = await rt.exec(code, timeout=timeout)
    txt = outs_text(outs)
    if any(o.get('output_type') == 'error' for o in outs): raise RuntimeError(txt.strip())
    parts = txt.split(tag)
    if len(parts) < 3: raise RuntimeError(f'Missing response marker in remote output: {txt[:300]}')
    return AttrDict(json.loads(parts[-2]))

@patch
async def sh(
    self:ColabRuntime,
    cmd:str|list, # Shell command string (`shell=True`) or argv list (`shell=False`)
    cwd:str=None, # Working directory on the runtime
    env:dict=None, # Extra environment variables merged with the runtime environment
    check:bool=True, # Raise `RuntimeError` when the command exits non-zero?
    timeout:float=None, # Execution timeout in seconds
)->AttrDict:
    "Run a shell command on the runtime and return `AttrDict(returncode, stdout, stderr)`."
    body = rf'''
import os, subprocess
_env = {{**os.environ, **{dict(env or {})!r}}} if {bool(env)} else None
_r = subprocess.run({cmd!r}, shell={isinstance(cmd, str)}, cwd={cwd!r}, env=_env, capture_output=True, text=True)
_res = dict(returncode=_r.returncode, stdout=_r.stdout, stderr=_r.stderr)
'''
    res = await _exec_json(self, body, timeout=timeout)
    if check and res.returncode: raise RuntimeError(f'Command failed ({res.returncode}):\n{res.stderr or res.stdout}'.strip())
    return res

Verify `sh` with a string command, an argv list, and `check=False` on a non-zero exit code:

In [ ]:
r_ok = await rt.sh('uname -sr && which uv')
assert r_ok.returncode == 0 and 'Linux' in r_ok.stdout
r_fail = await rt.sh(['false'], check=False)
test_eq(r_fail.returncode, 1)
r_ok.stdout.strip()

Colab does not publish guaranteed RAM or disk quotas, and hardware varies by region. `sys_info` queries the live machine in one call for Python version, CPU count, RAM, `/content` disk space, GPUs (`nvidia-smi`), and TPU settings.


In [ ]:
#| export
@patch
async def sys_info(self:ColabRuntime)->AttrDict:
    "Return live Python, CPU, RAM, disk, GPU, and TPU details for this runtime."
    body = r'''
import os, platform, shutil, socket, subprocess
def _mem_gb():
    info = {}
    with open('/proc/meminfo') as f:
        for ln in f:
            k, v = ln.split(':', 1)
            info[k] = int(v.split()[0]) / (1024 * 1024)
    return round(info.get('MemTotal', 0), 1), round(info.get('MemAvailable', 0), 1)
ram_tot, ram_avail = _mem_gb()
disk = shutil.disk_usage('/content')
gpus = []
if shutil.which('nvidia-smi'):
    r = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,memory.used', '--format=csv,noheader,nounits'], capture_output=True, text=True)
    if r.returncode == 0:
        for ln in r.stdout.strip().splitlines():
            name, tot, used = [p.strip() for p in ln.split(',')]
            gpus.append(dict(name=name, vram_gb=round(float(tot)/1024, 1), used_gb=round(float(used)/1024, 1)))
tpu = os.environ.get('TPU_NAME') or os.environ.get('COLAB_TPU_ADDR') or None
_res = dict(python=platform.python_version(), host=socket.gethostname(), cpus=os.cpu_count(),
    ram_gb=ram_tot, ram_avail_gb=ram_avail,
    disk_gb=round(disk.total / 1e9, 1), disk_free_gb=round(disk.free / 1e9, 1), gpus=gpus, tpu=tpu)
'''
    return await _exec_json(self, body)

In [ ]:
info = await rt.sys_info()
assert info.cpus >= 2 and info.ram_gb > 10 and info.disk_gb > 20
info

## Syncing directories

Uploading a directory tree file by file over `/api/contents` requires one HTTP request per directory and file. `upload_dir` packs the local directory into one `.tar.gz` archive, uploads it with `rt.upload` (reusing `00_core`'s 32 MB chunking), and extracts it remotely. `DEFAULT_EXCLUDES` skips Git metadata, virtual environments, caches, and `.DS_Store` files.


In [ ]:
#| export
DEFAULT_EXCLUDES = ('.git', '__pycache__', '.venv', '.ipynb_checkpoints', 'node_modules', '.DS_Store', '*.pyc')

def _excluded(rel_parts:tuple, patterns:tuple)->bool: return any(fnmatch.fnmatch(part, pat) for part in rel_parts for pat in patterns)

def _pack_local_dir(local_dir:Path, exclude:tuple=DEFAULT_EXCLUDES)->bytes:
    "Pack `local_dir` into an in-memory `.tar.gz`, skipping names matching `exclude`."
    root, buf = Path(local_dir).resolve(), io.BytesIO()
    if not root.is_dir(): raise NotADirectoryError(root)
    with tarfile.open(fileobj=buf, mode='w:gz') as tf:
        for p in sorted(root.rglob('*')):
            rel = p.relative_to(root)
            if _excluded(rel.parts, exclude): continue
            tf.add(p, arcname=rel.as_posix(), recursive=False)
    return buf.getvalue()

@patch
async def upload_dir(
    self:ColabRuntime,
    local_dir:str|Path, # Local directory to archive and upload
    remote_dir:str='/content/work', # Destination directory on the runtime
    exclude:tuple=DEFAULT_EXCLUDES, # Basename or glob patterns to skip
    clear:bool=False, # Remove existing contents of `remote_dir` before extracting?
    timeout:float=300, # Extraction timeout in seconds
)->str:
    "Archive `local_dir` into a `.tar.gz`, upload it once, and extract it into `remote_dir`."
    data = _pack_local_dir(Path(local_dir), exclude=exclude)
    tmp = f'/tmp/fc-up-{uuid.uuid4().hex}.tar.gz'
    await self.upload(data, tmp)
    body = rf'''
import os, shutil, tarfile
dst, tmp = {remote_dir!r}, {tmp!r}
try:
    if {bool(clear)} and os.path.exists(dst): shutil.rmtree(dst)
    os.makedirs(dst, exist_ok=True)
    with tarfile.open(tmp, 'r:gz') as tf: tf.extractall(dst, filter='data')
    _res = dict(ok=True, dst=dst)
finally:
    if os.path.exists(tmp): os.remove(tmp)
'''
    await _exec_json(self, body, timeout=timeout)
    return remote_dir

`download_dir` runs the reverse. It archives `remote_dir` into `/tmp`, fetches it via `rt.download`, and removes the remote archive. Local extraction uses `filter='data'` to keep paths inside `local_dir`.


In [ ]:
#| export
@patch
async def download_dir(
    self:ColabRuntime,
    remote_dir:str, # Directory on the runtime to archive and download
    local_dir:str|Path, # Local destination directory
    exclude:tuple=DEFAULT_EXCLUDES, # Basename or glob patterns to skip
    timeout:float=300, # Archiving timeout in seconds
)->Path:
    "Archive `remote_dir` on the runtime, download it once, and unpack into `local_dir`."
    tmp = f'/tmp/fc-down-{uuid.uuid4().hex}.tar.gz'
    body = rf'''
import fnmatch, os, tarfile
src, tmp, pats = {remote_dir!r}, {tmp!r}, {tuple(exclude)!r}
if not os.path.isdir(src): raise FileNotFoundError(src)
with tarfile.open(tmp, 'w:gz') as tf:
    for cur, dirs, files in os.walk(src):
        dirs[:] = [d for d in sorted(dirs) if not any(fnmatch.fnmatch(d, p) for p in pats)]
        for fn in sorted(files):
            if any(fnmatch.fnmatch(fn, p) for p in pats): continue
            full = os.path.join(cur, fn)
            tf.add(full, arcname=os.path.relpath(full, src))
_res = dict(size=os.path.getsize(tmp))
'''
    try:
        await _exec_json(self, body, timeout=timeout)
        raw = await self.download(tmp)
    finally: await self.sh(['rm', '-f', tmp], check=False)
    dst = Path(local_dir)
    dst.mkdir(parents=True, exist_ok=True)
    with tarfile.open(fileobj=io.BytesIO(raw), mode='r:gz') as tf: tf.extractall(dst, filter='data')
    return dst

Upload a nested local directory that also holds an ignored `__pycache__` file. Then write a new file remotely and download the directory back locally.


In [ ]:
td = tempfile.TemporaryDirectory(prefix='fastcolab-sync-')
src, dst = Path(td.name)/'src', Path(td.name)/'dst'
(src/'sub').mkdir(parents=True)
(src/'sub'/'hello.txt').write_text('hello from mac')
(src/'__pycache__').mkdir()
(src/'__pycache__'/'skip.pyc').write_bytes(b'ignore')
await rt.upload_dir(src, '/content/sync_demo', clear=True)
(await rt.sh('find /content/sync_demo -type f | sort')).stdout.strip()

In [ ]:
await rt.sh(['python3', '-c', "open('/content/sync_demo/result.txt', 'w').write('computed on colab')"])
await rt.download_dir('/content/sync_demo', dst)
test_eq((dst/'sub'/'hello.txt').read_text(), 'hello from mac')
test_eq((dst/'result.txt').read_text(), 'computed on colab')
assert not (dst/'__pycache__').exists()
synced = sorted(p.relative_to(dst).as_posix() for p in dst.rglob('*') if p.is_file())
td.cleanup()
synced

## Running local scripts and notebooks remotely

Running a local `.py` file on a Colab kernel needs four small adjustments. `_script_payload` strips any `#!` shebang, sets `__file__` and `sys.argv`, applies temporary `os.environ` values before restoring them, and treats `SystemExit(0)` as a clean exit rather than a traceback. `ColabRuntime.run_script` parses the exit marker and raises on non-zero exits when `check=True`.


In [ ]:
#| export
def _strip_shebang(src:str)->str:
    return src.split('\n', 1)[1] if src.startswith('#!') and '\n' in src else ('' if src.startswith('#!') else src)

def _script_payload(src:str, name:str, tag:str, args:list=None, env:dict=None)->str:
    "Wrap Python script source so `__file__`, `sys.argv`, `os.environ`, and `SystemExit` behave like CLI execution."
    body = _strip_shebang(src)
    argv = [name, *(str(a) for a in (args or []))]
    return rf'''
import json as _fc_j, os as _fc_os, sys as _fc_sys
_fc_old_argv = _fc_sys.argv[:]
_fc_old_env = {{k: _fc_os.environ.get(k) for k in {list((env or {}).keys())!r}}}
_fc_rc = 0
try:
    _fc_sys.argv = {argv!r}
    for _k, _v in {dict(env or {})!r}.items(): _fc_os.environ[_k] = str(_v)
    _fc_ns = {{'__name__': '__main__', '__file__': {name!r}}}
    exec(compile({body!r}, {name!r}, 'exec'), _fc_ns)
except SystemExit as _e:
    if _e.code is None: _fc_rc = 0
    elif isinstance(_e.code, int): _fc_rc = _e.code
    else:
        print(_e.code, file=_fc_sys.stderr)
        _fc_rc = 1
finally:
    _fc_sys.argv = _fc_old_argv
    for _k, _v in _fc_old_env.items():
        if _v is None: _fc_os.environ.pop(_k, None)
        else: _fc_os.environ[_k] = _v
print({tag!r} + _fc_j.dumps(dict(returncode=_fc_rc)) + {tag!r})
'''

@patch
async def run_script(
    self:ColabRuntime,
    script:str|Path, # Local path to a Python script
    args:list=None, # Command-line arguments exposed as `sys.argv[1:]`
    env:dict=None, # Temporary environment variables for the script
    check:bool=True, # Raise `RuntimeError` on unhandled exception or non-zero `SystemExit`?
    timeout:float=None, # Execution timeout in seconds
)->AttrDict:
    "Run a local Python script on this runtime and return `AttrDict(returncode, text, outputs)`."
    p = Path(script)
    tag = f'__FC_EXIT_{uuid.uuid4().hex}__'
    outs = await self.exec(_script_payload(p.read_text(), p.name, tag, args=args, env=env), timeout=timeout)
    raw = outs_text(outs)
    if any(o.get('output_type') == 'error' for o in outs):
        if check: raise RuntimeError(raw.strip())
        return AttrDict(returncode=1, text=raw, outputs=outs)
    parts = raw.split(tag)
    rc = json.loads(parts[-2])['returncode'] if len(parts) >= 3 else 0
    clean = (parts[0] + parts[-1]).rstrip('\n') if len(parts) >= 3 else raw.rstrip('\n')
    for o in outs:
        if o.get('output_type') == 'stream' and tag in o.get('text', ''):
            ps = o['text'].split(tag)
            o['text'] = ps[0] + ps[-1].lstrip('\n')
    if check and rc: raise RuntimeError(f'{p.name} exited with code {rc}:\n{clean}'.strip())
    return AttrDict(returncode=rc, text=clean, outputs=outs)

Run a local script that starts with a shebang, reads `sys.argv` and an environment variable, and exits explicitly via `sys.exit(0)`:

In [ ]:
with tempfile.TemporaryDirectory(prefix='fastcolab-script-') as td:
    sp = Path(td)/'calc.py'
    sp.write_text('#!/usr/bin/env python3\nimport os, sys\nprint(int(sys.argv[1]) * int(os.environ["MULT"]))\nsys.exit(0)\n')
    sr = await rt.run_script(sp, args=['7'], env={'MULT': '6'})
test_eq(sr.returncode, 0)
test_eq(sr.text.strip(), '42')
sr

`run_nb` opens a fresh remote kernel and executes each code cell of a local notebook in order. It writes the real Jupyter outputs back onto every cell, shuts down the temporary kernel on exit, and optionally saves the result to `out_path`.


In [ ]:
#| export
@patch
async def run_nb(
    self:ColabRuntime,
    nb_path:str|Path, # Local `.ipynb` path to execute remotely
    out_path:str|Path=None, # Optional output `.ipynb` path (can match `nb_path` to save in place)
    allow_errors:bool=False, # Continue running later cells after a cell raises?
    timeout:float=None, # Per-cell execution timeout in seconds
)->Notebook:
    "Execute a local notebook on a fresh remote kernel, storing outputs on each code cell."
    nb = read_nb(nb_path)
    async with await self.kernel() as kc:
        for i, cell in enumerate(nb.cells):
            if cell.cell_type != 'code' or not cell.source.strip(): continue
            outs = await kc.exec_outs(cell.source, timeout=timeout)
            cell.outputs = outs
            cell.execution_count = i + 1
            if not allow_errors and any(o.get('output_type') == 'error' for o in outs):
                if out_path: write_nb(nb, out_path)
                raise RuntimeError(f'Cell {i} failed:\n{outs_text(outs).strip()}')
    if out_path: write_nb(nb, out_path)
    return nb

Create a two-cell notebook locally and run it on `rt`. Inspect the saved output:


In [ ]:
with tempfile.TemporaryDirectory(prefix='fastcolab-nb-') as td:
    inp, outp = Path(td)/'in.ipynb', Path(td)/'out.ipynb'
    write_nb(new_nb([mk_cell('x = 21'), mk_cell('x * 2')]), inp)
    executed = await rt.run_nb(inp, out_path=outp)
    saved = read_nb(outp)
    res_txt = outs_text(saved.cells[1].outputs).strip()
test_eq(res_txt, '42')
res_txt

`Colab.run_script` matches `colab run`. It provisions a runtime, uploads optional files or directories, executes the script, and downloads requested artifacts. A `finally` block deletes the runtime unless `keep=True`.


In [ ]:
#| export
@patch
async def run_script(
    self:Colab,
    script:str|Path, # Local path to a Python script
    hw:str|dict='cpu', # Hardware alias or raw `runtimeSpec` dict
    high_mem:bool=True, # Request `SHAPE_HIGHMEM` when supported?
    args:list=None, # Command-line arguments exposed as `sys.argv[1:]`
    env:dict=None, # Temporary environment variables for the script
    upload:dict=None, # Mapping `{local_path: remote_path}` of files or directories to upload first
    download:dict=None, # Mapping `{remote_path: local_path}` of files or directories to download after
    keep:bool=False, # Keep the runtime alive after finishing instead of deleting it?
    check:bool=True, # Raise `RuntimeError` on non-zero exit?
    timeout:float=None, # Execution timeout in seconds
)->AttrDict:
    "Provision a runtime, run a local script with optional uploads and artifact downloads, and clean up."
    rt = await self.start(hw, high_mem=high_mem)
    try:
        for lp, rp in (upload or {}).items():
            p = Path(lp)
            if p.is_dir(): await rt.upload_dir(p, rp)
            else: await rt.upload(p, rp)
        res = await rt.run_script(script, args=args, env=env, check=check, timeout=timeout)
        for rp, lp in (download or {}).items():
            if rp.endswith('/'): await rt.download_dir(rp.rstrip('/'), lp)
            else: await rt.download(rp, lp)
        if keep: res['runtime'] = rt
        return res
    finally:
        if not keep: await rt.stop()

Run an ephemeral job on a fresh CPU runtime. This uploads `in.txt`, transforms it remotely, downloads `out.txt`, and deletes the runtime on exit.


In [ ]:
with tempfile.TemporaryDirectory(prefix='fastcolab-run-') as td:
    td = Path(td)
    (td/'in.txt').write_text('fastcolab')
    sp = td/'job.py'
    sp.write_text("from pathlib import Path\nPath('/content/out.txt').write_text(Path('/content/in.txt').read_text().upper())\nprint('done')\n")
    ephem = await c.run_script(sp, hw='cpu', upload={td/'in.txt': '/content/in.txt'}, download={'/content/out.txt': td/'out.txt'})
    artifact = (td/'out.txt').read_text()
test_eq(ephem.returncode, 0)
test_eq(artifact, 'FASTCOLAB')
dict(text=ephem.text, artifact=artifact)

## Cleaning up runtimes

`stop_all` deletes active runtimes on your account in one call. Pass `ids` to target specific runtimes, or `dry_run=True` to preview what would stop without deleting anything.


In [ ]:
#| export
@patch
async def stop_all(
    self:Colab,
    ids:list=None, # Optional subset of runtime IDs or resource names to delete (`None` means all)
    dry_run:bool=False, # Return matched runtime IDs without deleting them?
)->L:
    "Delete active Colab runtimes on this account and return the matched runtime IDs."
    rts = await self.runtimes()
    if ids is not None:
        want = {i.split('/')[-1] for i in ids}
        rts = rts.filter(lambda r: r.id in want)
    if not dry_run: await asyncio.gather(*(self.delete(r.name, allow_missing=True) for r in rts))
    return rts.attrgot('id')

Preview `stop_all` with `dry_run=True`. Then delete only this notebook's runtime (`rt`) and verify it is gone.


In [ ]:
matched = await c.stop_all(ids=[rt.id], dry_run=True)
test_eq(list(matched), [rt.id])
stopped = await c.stop_all(ids=[rt.id])
test_eq(list(stopped), [rt.id])
remaining = await c.runtimes()
assert all(r.id != rt.id for r in remaining)
stopped

In [ ]:
#| hide
#| eval: false
import nbdev; nbdev.nbdev_export()